> 📓 **Lesson 1.8 — Part 1 of 4: Descriptive Statistics — Summarising Your Data**
>
> This notebook was split out of the original single `eda_basic.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.8.
>
> Other notebooks in this set: `Part_2_data_quality.ipynb`, `Part_3_data_transformation.ipynb`, `Part_4_reading_writing_data.ipynb`

# Lesson 1.8: EDA Basic

Welcome to Exploratory Data Analysis. This notebook takes one raw, messy business file and walks the
full path: understanding its structure, cleaning it, transforming it, and moving it in and out of
files.

**Structure — the four learning outcomes, in order:**
* **Part 1: Descriptive Statistics** — *summarise* a dataset: shape, data types, distributions.
* **Part 2: Data Quality** — *handle* the messy reality: missing values, duplicates, impossible values.
* **Part 3: Data Transformation** — *transform* for analysis: mapping, labels, strings, categories, dates.
* **Part 4: Reading & Writing Data** — *read and write* CSV, JSON, Excel, databases.

**For Learners:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 150 minutes.** One messy file, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Descriptive Statistics | **Summarise** a dataset: shape, dtypes, distributions | 33 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Quality | **Handle** missing values, duplicates, impossible values | 42 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Data Transformation | **Transform**: mapping, labels, strings, categories, dates, grouping | 35 min |
> | **Part 4** | Reading & Writing Data | **Read and write** CSV, JSON, Excel, databases | 15 min |
>
> **The spine:** we work on one file, `data/cafe_june_raw.csv`, from start to finish. Each section
> improves the same `clean` table, and Part 4 saves it. Small hand-built tables appear alongside it
> as *drills* — they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`;
> the Appendix at the end is self-study.


### The business problem

> **The Daily Grind** is a four-outlet café chain in Singapore. Revenue has been flat for two
> quarters, and the owner has to decide whether to renew the Marina Bay lease. She asks her
> assistant to send you the sales data. What arrives is a **raw till export**: one row per outlet,
> per day, per part of the day, straight out of the point-of-sale system, untouched.
>
> Nobody can answer the owner's question from this file yet. Today's job is to make it
> answerable — and to be able to say *why* every number in it can be trusted.

This is the first of three lessons on the same problem:

| Lesson | The question | What you do |
|---|---|---|
| **1.8 — today** | **Can I trust this data?** | clean one month of the raw export |
| 1.9 | What is the pattern? | 18 months, cleaned: time, joins, grouping |
| 1.10 | How do I make them act? | one chart, one slide, one decision |


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Load the two toolkits we need. `pd` and `np` are just short nicknames so we can
#    type `pd.something` instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np


In [ ]:
# 👉 Load the dataset we will use all session: June 2025's till export from four cafés.
#    `read_csv` reads a comma-separated text file into a DataFrame -- a table with named columns.
#    pandas already treats an empty field, "NA" and "n/a" as missing.
raw = pd.read_csv("../data/cafe_june_raw.csv")

raw


### 🎬 Why this matters — before you trust a single number

Run the next three cells. The chain has **four** cafés and its busiest shift takes about \$1,000.


In [ ]:
# 👉 `.value_counts()` counts how many rows have each value. How many cafés do you count?
raw["outlet"].value_counts()


In [ ]:
# 👉 The same question of the daypart column. There are three parts to a trading day.
raw["daypart"].value_counts()


In [ ]:
# 👉 Sort the takings column and look at the two ends. `.dropna()` skips the blank cells,
#    because a sort cannot compare text with a blank -- which is itself a clue.
#    `.iloc[[0, -1]]` takes the first and last rows of the sorted result.
raw["revenue_raw"].dropna().sort_values().iloc[[0, -1]]


**Three problems, in three lines of output.**

1. **Twelve spellings for four cafés.** `Raffles Place`, `raffles place`, `RAFFLES PLACE`,
   `Raffles Pl.`… Group by outlet today and you get twelve cafés, four of which are the same shop.
2. **Nine labels for three dayparts** — `Morning`, `morning`, `AM`, and so on.
3. **The revenue column is not a number.** Sorted, the "smallest" value is `" 1,006.71 "` and the
   "largest" is `"S$94.41"`, because pandas is comparing them as **text**: a space sorts before a
   digit, and the letter `S` sorts after every digit. Sorted as text, \$98,000 loses to \$99.

Any average, chart or model built on this file is wrong before you start. Worse, none of it would
*look* wrong: it would produce numbers, with decimal places, and nobody in the meeting would know.

Part 1 is the routine that finds problems like these in about two minutes.


---

## Part 1: Descriptive Statistics — Summarising Your Data

**Learning outcome 1:** *Summarise a dataset using descriptive statistics and identify its shape,
data types, and distributions.*

**Goal:** know what you are holding before you touch it.

⏱️ ~33 min including Group Exercise 1


### 1.1: The First Look — a five-move first look

Do these five, in this order, every time you meet a new file. It takes two minutes and it is the
difference between analysis and guesswork.

| Move | Question it answers |
|---|---|
| `.head()` | What do the rows actually look like? |
| `.shape` | How big is it? |
| `.info()` | What type is each column, and where are the holes? |
| `.dtypes` | Is anything stored as the wrong type? |
| `.describe()` | Are the numbers plausible? |


In [ ]:
# 👉 Move 1: look at real rows. `.head()` shows the first 5 (pass a number for more).
#    Never skip this. Half of all data problems are visible to the naked eye.
raw.head()


In [ ]:
# 👉 Move 2: how big? `.shape` gives (rows, columns). No brackets -- it is a value, not a method.
raw.shape


In [ ]:
# 👉 Move 3: the single most useful command in pandas. For every column it reports the
#    non-null count and the type. Compare each count with the row count above: the gaps are holes.
raw.info()


In [ ]:
# 👉 Move 4: just the types. `object` means text (or mixed). Note `revenue_raw` and `date_text`
#    are text, not numbers and dates -- and `tickets` is a decimal, which is odd for a count.
raw.dtypes


In [ ]:
# 👉 Move 5: the numbers. Read the min and max rows first: that is where impossible values hide.
#    Notice which columns are MISSING from this table -- `.describe()` only sees numeric ones.
raw.describe()


In [ ]:
# 👉 `.describe()` skips text columns by default. Ask for them explicitly and you get
#    count / unique / top / freq -- which is where spelling variants show up.
raw.describe(include="object")


**Write down what the first look found** — this is our to-do list for Part 2:

1. **Missing values** — `revenue_raw` 3, `tickets` 2, `items` 3, `staff_on_shift` 5,
   `manager_email` 2, `notes` 332.
2. **Duplicate rows** — 366 rows, but June has only 30 dates (`date_text` shows 30 unique values),
   and 30 days × 4 cafés × 3 dayparts = **360** possible shifts. Six rows too many: something was
   sent twice.
3. **Wrong types** — revenue is text; the date is text; counts are decimals.
4. **Impossible values** — `tickets` has a minimum of **-4**, and a shift with **0** tickets that
   still took money.
5. **Twelve outlet spellings and nine daypart labels** for four cafés and three dayparts.

Five problems, five different right answers. That is Part 2 and Part 3.

> **Why `tickets` is a decimal.** A column of whole numbers with even one missing value cannot stay
> an integer, because there is no integer that means "missing". pandas quietly promotes the whole
> column to `float64`. A count stored as a decimal is therefore a *symptom*: it usually means the
> column has holes in it.


### 1.2a: Those Statistics, Unpacked on Our Data

`.describe()` is a bundle of simpler methods. Here we take them one at a time — on a slice of
`raw` small enough that you can check the arithmetic by hand.


In [ ]:
# 👉 `.loc[[...], [...]]` takes the listed rows and just two numeric columns. Small on purpose:
#    8 rows you can add up yourself. (`revenue_raw` cannot appear here -- it is still text.)
#    Row 12 is in the list deliberately: its ticket count is missing.
peek = raw.loc[[0, 1, 2, 3, 4, 5, 12, 13], ["tickets", "items"]]

peek


**Reductions** — a whole column in, one number out. By default they work *down* the rows.

Note the hole in row 12 of `tickets`: watch what each method does with it.


In [ ]:
# 👉 Total tickets sold and total items bought, across these 8 shifts.
#    One number per column comes back.
peek.sum()


In [ ]:
# 👉 The averages. `tickets` has 7 values, not 8, so `.mean()` divides by 7 -- it divides by
#    the number of values *present*. That is either exactly what you want or a silent bug,
#    depending on why the value is missing.
peek.mean()


**`skipna` — what to do about the hole.** By default pandas ignores missing values and
carries on. That is convenient, and it is also how a hole becomes invisible.


In [ ]:
# 👉 `skipna=False` refuses to silently work around the hole. `tickets` becomes NaN;
#    `items` still has a real total. Now you can *see* which column had a gap.
peek.sum(skipna=False)


**Indirect statistics:** finding *where* the max or min value sits (the row label).


In [ ]:
# 👉 Not the biggest *value* -- the row *label* where it sits. Combine it with `.loc[...]`
#    to pull the whole row out and look at the shift that did it.
peek["items"].idxmax()


In [ ]:
# 👉 Same for the minimum, on both columns at once.
peek.idxmin()


**Accumulations:** running totals.


In [ ]:
# 👉 A running total: each row is itself plus everything above it. Read the bottom row of
#    `items` -- that is the same number `.sum()` gave you.
peek.cumsum()


**`.describe()` — the bundle.** Everything above, in one call. You met it as move 5 of the first look.


In [ ]:
# 👉 All of the above in one call, plus the quartiles. This is why `.describe()` is move 5
#    and not five separate commands.
peek.describe()


**Categorical / non-numeric data:** `.describe()` behaves differently for text, showing counts
and the most frequent value rather than means.


In [ ]:
# 👉 A Series is a single column of data. This one holds text, so `.describe()` switches to
#    count / unique / top / freq. `top` is the most common value; `freq` is how often it appears.
raw["outlet"].describe()


In [ ]:
# 👉 List the distinct values, in the order they first appear. Duplicates are dropped.
raw["outlet"].unique()


In [ ]:
# 👉 Count how many times each distinct value appears, most frequent first.
raw["outlet"].value_counts()


### 1.2b: The `axis` Sandbox — a deliberately meaningless table

One piece of notation left: **`axis`**. Every reduction can run *down* the rows (the default) or
*across* the columns. This drill uses a made-up table, because on real data one of the two
directions is usually nonsense — and it is easier to see the mechanics when nothing is at stake.


In [ ]:
# 👉 Build a small table by hand. The outer [ ] is a list of rows; each inner [ ] is one row.
#    `np.nan` is how you write a missing value.
demo = pd.DataFrame(
    [[1.4, np.nan], [7.1, -4.5], [np.nan, np.nan], [0.75, -1.3]],
    index=["a", "b", "c", "d"],
    columns=["one", "two"],
)

demo


In [ ]:
# 👉 Add up each column, top to bottom. One number comes back per column.
demo.sum()


In [ ]:
# 👉 Same addition, but sideways: add across each row. `axis="columns"` means 'go across'.
demo.sum(axis="columns")


In [ ]:
# 👉 `skipna=False` says 'do NOT ignore missing values'. Any NaN in a column poisons its total.
demo.sum(skipna=False)


In [ ]:
# 👉 Same idea going across the rows instead of down the columns.
demo.sum(axis=1, skipna=False)


**Back to reality.** On our café export, `axis="columns"` is almost always wrong —
`tickets + staff_on_shift` is not a quantity that exists. Adding *down* a column
("total tickets in June") is the meaningful direction. Know both; reach for the first.


### 🛠️ Group Exercise 1 — Summarising (8 min)

Which shift sold the most items? Use `.idxmax()` on `raw["items"]` to get the row label, then `.loc[...]` to pull that whole row out.

*Expected:* one row — the morning of Tuesday 10 June at Raffles Place, with 336 items.

---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Which shift sold the most items?**

In [ ]:
# ===== 🏪 REAL DATASET (`raw`) — our coffee-shop data =====
# 👉 `.idxmax()` gives the *row label* of the largest value (not the value itself),
#    then `.loc[...]` pulls that whole row out so we can see which shift it was.
busiest = raw["items"].idxmax()

raw.loc[busiest]


In [ ]:
# 👉 Same answer as a one-row table rather than a Series: pass a *list* of labels.
raw.loc[[raw["items"].idxmax()]]


---

# ☕ Break — 10 minutes

**Where we are:** you can now describe *what* a dataset looks like.
**Next up:** Part 2 — fixing what is wrong with it (missing values, duplicates, impossible values).


📂 **Open** `Part_2_data_quality.ipynb` to continue.